
# RQ2 Formal Experiment 1 — Fed-CGM

In [1]:

from pathlib import Path
import sys, json, random
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/dissertation")
else:
    PROJECT_ROOT = Path("../../..").resolve()

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import dataloader as data
from src import federated as fed
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

TUNING_DIR = (
    PROJECT_ROOT / "experiments" / "7_federated_learning" / "results"
    / "configuration_tuning" / "fed_cgm_multiseed"
)
SELECTED_CONFIG_PATH = TUNING_DIR / "selected_fl_configuration.json"
assert SELECTED_CONFIG_PATH.exists(), (
    f"Locked FL configuration not found: {SELECTED_CONFIG_PATH}"
)

with open(SELECTED_CONFIG_PATH) as f:
    FL_CONFIG = json.load(f)

LOCAL_EPOCHS = int(FL_CONFIG["local_epochs"])
CLIENTS_PER_ROUND = int(FL_CONFIG["clients_per_round"])
EARLY_STOP_PATIENCE = int(FL_CONFIG["early_stop_patience"])
MAX_ROUNDS = int(FL_CONFIG["max_rounds"])
LR_PATIENCE = int(FL_CONFIG["lr_patience"])
LR_FACTOR = float(FL_CONFIG["lr_factor"])
LR = float(FL_CONFIG["learning_rate"])
WEIGHT_DECAY = float(FL_CONFIG["weight_decay"])
BATCH_SIZE = int(FL_CONFIG["batch_size"])
MAX_GRAD_NORM = float(FL_CONFIG["max_grad_norm"])

GRU_HIDDEN_SIZE = int(FL_CONFIG["gru_hidden_size"])
GRU_NUM_LAYERS = int(FL_CONFIG["gru_num_layers"])
GRU_DROPOUT = float(FL_CONFIG["gru_dropout"])

assert (LOCAL_EPOCHS, CLIENTS_PER_ROUND, EARLY_STOP_PATIENCE) == (5, 15, 20), (
    "The locked configuration is not the expected E5_C15_P20. "
    f"Found E{LOCAL_EPOCHS}_C{CLIENTS_PER_ROUND}_P{EARLY_STOP_PATIENCE}."
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Locked FL protocol:",
      f"E={LOCAL_EPOCHS}, C={CLIENTS_PER_ROUND}/30, P={EARLY_STOP_PATIENCE}")
print("LR:", LR, "WD:", WEIGHT_DECAY, "MAX_ROUNDS:", MAX_ROUNDS)

from src.models.nn_baseline import GRUPredictor


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/dissertation
Device: cuda
Locked FL protocol: E=5, C=15/30, P=20
LR: 0.0001 WD: 1e-05 MAX_ROUNDS: 100


## 1. Load aligned data

In [2]:

stats = data.load_cgm_stats()

client_loaders = fed.load_federated_clients(
    "train", encoder_name=None, batch_size=BATCH_SIZE
)
val_loader = data.load_multimodal_tensors(
    "val", encoder_name=None, batch_size=BATCH_SIZE, shuffle=False
)
test_loader = data.load_multimodal_tensors(
    "test", encoder_name=None, batch_size=BATCH_SIZE, shuffle=False
)
test_participant_ids = data.load_multimodal_participant_ids("test")

aligned_train_samples = data.load_multimodal_samples("train")
assert len(client_loaders) == 30
assert len(aligned_train_samples) == sum(len(x.dataset) for x in client_loaders.values())
assert not any(s["has_meal"] and not s["has_image"] for s in aligned_train_samples)

print("Aligned train samples:", len(aligned_train_samples))
print("Train clients:", len(client_loaders))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))
print("Alignment checks: PASS")


Aligned train samples: 82681
Train clients: 30
Validation samples: 16355
Test samples: 16818
Alignment checks: PASS


## 2. Model and forward function

In [3]:

def model_fn():
    return GRUPredictor(
        hidden_size=GRU_HIDDEN_SIZE,
        num_layers=GRU_NUM_LAYERS,
        dropout=GRU_DROPOUT,
        num_horizons=len(HORIZONS),
    )

def forward_fn(model, batch, device):
    cgm, tsm, has_meal, target = batch
    return model(cgm.to(device)), target.to(device)

m = model_fn()
print(m)
print("Trainable parameters:", sum(p.numel() for p in m.parameters() if p.requires_grad))


GRUPredictor(
  (gru): GRU(1, 256, num_layers=3, batch_first=True, dropout=0.2)
  (head): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=5, bias=True)
  )
)
Trainable parameters: 1021957


## 3. Output paths and helpers

In [4]:

EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "7_federated_learning"
RESULTS_DIR = EXPERIMENT_DIR / "results" / "fed_cgm_only"
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints" / "fed_cgm_only"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RUN_RESULTS_PATH = RESULTS_DIR / "run_results.json"
SUMMARY_PATH = RESULTS_DIR / "summary.csv"

assert str(RESULTS_DIR).startswith(str(PROJECT_ROOT))
assert str(CHECKPOINT_DIR).startswith(str(PROJECT_ROOT))

print("Results:", RESULTS_DIR)
print("Checkpoints:", CHECKPOINT_DIR)


def to_jsonable_results(result_dict):
    return {str(k): v for k, v in result_dict.items()}

def to_jsonable_per_participant(result_dict):
    return {
        str(pid): {str(h): v for h, v in per_h.items()}
        for pid, per_h in result_dict.items()
    }

def evaluate_model(model, loader, participant_ids=None):
    y_pred, y_true = train.predict(model, loader, device=DEVICE, forward_fn=forward_fn)
    overall = metrics.evaluate(
        y_pred, y_true, stats["cgm_mean"], stats["cgm_std"]
    )
    per_participant = None
    if participant_ids is not None:
        per_participant = metrics.evaluate_per_participant(
            y_pred, y_true, participant_ids,
            stats["cgm_mean"], stats["cgm_std"]
        )
    return overall, per_participant

def summarise_metric(results, split, metric_name):
    rows = []
    for h in HORIZONS:
        values = np.asarray([
            float(r[f"{split}_results"][str(h)][metric_name])
            for r in results
        ])
        rows.append({
            "horizon_min": h,
            "mean": values.mean(),
            "sd": values.std(ddof=1),
        })
    return pd.DataFrame(rows)


Results: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_only
Checkpoints: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/checkpoints/fed_cgm_only


## 4. Train/resume 10 formal seeds

In [5]:

all_results = []
if RUN_RESULTS_PATH.exists():
    with open(RUN_RESULTS_PATH) as f:
        all_results = json.load(f)

completed = {int(r["seed"]) for r in all_results}
print("Already completed:", sorted(completed))

for seed in SEEDS:
    if seed in completed:
        print(f"SKIP seed={seed}: already saved on Drive")
        continue

    print("\n" + "=" * 80)
    print(f"Fed-CGM | seed={seed}")
    print("=" * 80)

    fed.set_seed(seed)
    seed_ckpt = CHECKPOINT_DIR / f"seed{seed}"

    model, history = fed.federated_train(
        model_fn=model_fn,
        client_loaders=client_loaders,
        val_loader=val_loader,
        cgm_mean=stats["cgm_mean"],
        cgm_std=stats["cgm_std"],
        device=DEVICE,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        local_epochs=LOCAL_EPOCHS,
        max_rounds=MAX_ROUNDS,
        clients_per_round=CLIENTS_PER_ROUND,
        early_stop_patience=EARLY_STOP_PATIENCE,
        lr_patience=LR_PATIENCE,
        lr_factor=LR_FACTOR,
        checkpoint_dir=seed_ckpt,
        verbose=False,
        progress_bar=True,
        progress_desc=f"Fed-CGM seed={seed}",
        forward_fn=forward_fn,
    )

    # federated_train returns the best-validation global model.
    val_results, _ = evaluate_model(model, val_loader)
    test_results, test_pp = evaluate_model(
        model, test_loader, test_participant_ids
    )

    best_round = int(np.argmin(history["val_loss"]) + 1)
    result = {
        "model": "Fed-CGM",
        "seed": int(seed),
        "fl_config": {
            "local_epochs": LOCAL_EPOCHS,
            "clients_per_round": CLIENTS_PER_ROUND,
            "early_stop_patience": EARLY_STOP_PATIENCE,
            "max_rounds": MAX_ROUNDS,
            "lr_patience": LR_PATIENCE,
            "lr_factor": LR_FACTOR,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "batch_size": BATCH_SIZE,
            "max_grad_norm": MAX_GRAD_NORM,
        },
        "n_rounds_trained": len(history["val_loss"]),
        "best_round": best_round,
        "best_val_loss": float(np.min(history["val_loss"])),
        "val_results": to_jsonable_results(val_results),
        "test_results": to_jsonable_results(test_results),
        "test_results_per_participant": to_jsonable_per_participant(test_pp),
    }

    all_results.append(result)
    completed.add(seed)

    # Save immediately after every seed.
    tmp = RUN_RESULTS_PATH.with_suffix(".tmp.json")
    with open(tmp, "w") as f:
        json.dump(all_results, f, indent=2)
    tmp.replace(RUN_RESULTS_PATH)

    print(f"SAVED seed={seed} to Drive")
    print("best round:", best_round)
    print("test RMSE:",
          [round(float(result["test_results"][str(h)]["rmse"]), 3) for h in HORIZONS])

print("\nCompleted seeds:", sorted(completed))


Already completed: []

Fed-CGM | seed=7
device: cuda  model: GRUPredictor  params: 1,021,957  trainable: 1,021,957  clients: 30 (15/30 sampled per round)


Fed-CGM seed=7:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED seed=7 to Drive
best round: 16
test RMSE: [14.174, 22.577, 33.014, 39.478, 43.83]

Fed-CGM | seed=14
device: cuda  model: GRUPredictor  params: 1,021,957  trainable: 1,021,957  clients: 30 (15/30 sampled per round)


Fed-CGM seed=14:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED seed=14 to Drive
best round: 62
test RMSE: [12.471, 20.855, 31.814, 38.951, 43.4]

Fed-CGM | seed=21
device: cuda  model: GRUPredictor  params: 1,021,957  trainable: 1,021,957  clients: 30 (15/30 sampled per round)


...

SAVED seed=56 to Drive
best round: 52
test RMSE: [12.557, 20.996, 31.805, 38.797, 43.082]

Fed-CGM | seed=63
device: cuda  model: GRUPredictor  params: 1,021,957  trainable: 1,021,957  clients: 30 (15/30 sampled per round)


Fed-CGM seed=63:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED seed=63 to Drive
best round: 44
test RMSE: [12.825, 21.25, 32.302, 39.278, 43.635]

Fed-CGM | seed=70
device: cuda  model: GRUPredictor  params: 1,021,957  trainable: 1,021,957  clients: 30 (15/30 sampled per round)


Fed-CGM seed=70:   0%|          | 0/100 [00:00<?, ?it/s]

SAVED seed=70 to Drive
best round: 56
test RMSE: [12.639, 20.802, 31.782, 38.627, 43.058]

Completed seeds: [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]


## 5. Aggregate formal test results

In [6]:

with open(RUN_RESULTS_PATH) as f:
    all_results = json.load(f)

assert {int(r["seed"]) for r in all_results} == set(SEEDS), (
    "Not all 10 formal seeds are complete."
)

metric_tables = []
for metric_name in ["rmse", "mae"]:
    df = summarise_metric(all_results, "test", metric_name)
    df.insert(0, "metric", metric_name)
    metric_tables.append(df)

summary_df = pd.concat(metric_tables, ignore_index=True)
summary_df.to_csv(SUMMARY_PATH, index=False)

display(summary_df)

print("\nFed-CGM test RMSE mean ± SD:")
rmse_df = summary_df[summary_df["metric"] == "rmse"]
for _, row in rmse_df.iterrows():
    print(f'{int(row["horizon_min"]):>3} min: {row["mean"]:.3f} ± {row["sd"]:.3f}')

print("\nSaved:", SUMMARY_PATH)
print("Formal Fed-CGM complete.")


,metric,horizon_min,mean,sd
0,rmse,15,12.802032,0.578704
1,rmse,30,21.116316,0.576151
2,rmse,60,31.948941,0.454371
3,rmse,90,38.857043,0.422217
4,rmse,120,43.203549,0.467451
5,mae,15,8.597089,0.369007
6,mae,30,13.984616,0.344328
7,mae,60,20.748905,0.335936
8,mae,90,25.325586,0.420741
9,mae,120,28.394461,0.493934



Fed-CGM test RMSE mean ± SD:
 15 min: 12.802 ± 0.579
 30 min: 21.116 ± 0.576
 60 min: 31.949 ± 0.454
 90 min: 38.857 ± 0.422
120 min: 43.204 ± 0.467

Saved: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/fed_cgm_only/summary.csv
Formal Fed-CGM complete.
